<a href="https://colab.research.google.com/github/404sidrahfatima-spec/SOUP-ASSIGNMENT-/blob/main/SOUP_ASSIGNMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Align torchvision and torchaudio to the matching PyTorch version
!pip install -q --no-deps torchvision torchaudio --upgrade --extra-index-url https://download.pytorch.org/whl/cu132 || pip uninstall -y torchaudio torchvision

# 2. Install PEFT, datasets, bitsandbytes without upgrading torch
!pip install -q --no-deps peft
!pip install -q datasets accelerate bitsandbytes safetensors uv

# 3. Create soup environment cleanly
!mkdir -p logs out data base
!rm -rf /content/soupenv
!uv venv --no-managed-python --python 3.12 /content/soupenv
!nohup sh -c 'uv pip install --python /content/soupenv/bin/python "soup-cli[train]"' > logs/soup_install.log 2>&1 &

# 4. Verify cleanly
import torch
import transformers
import peft

print("CUDA Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print(f"torch: {torch.__version__} | transformers: {transformers.__version__} | peft: {peft.__version__}")

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: soupenv
Activate with: source soupenv/bin/activate


In [1]:
import torch
import transformers
import peft

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Allocated VRAM (MB):", round(torch.cuda.memory_allocated() / (1024**2), 2))
print(f"torch: {torch.__version__} | transformers: {transformers.__version__} | peft: {peft.__version__}")

CUDA Available: True
Device Name: Tesla T4
Allocated VRAM (MB): 0.0
torch: 2.15.0+cu132 | transformers: 5.22.2 | peft: 0.21.2


In [2]:
%%writefile common.py
import os, re, json, time, random, datetime
import numpy as np
import torch

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAXLEN = 512
MAX_NEW = 384
SEED = 42
N_TRAIN = 1500
N_EVAL = 200

def log(msg):
    print(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] {msg}", flush=True)

def seed_all(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def get_tokenizer():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id
    return tok

def load_model(path, dtype=torch.float16, **kw):
    from transformers import AutoModelForCausalLM
    try:
        return AutoModelForCausalLM.from_pretrained(path, dtype=dtype, attn_implementation="sdpa", **kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(path, torch_dtype=dtype, attn_implementation="sdpa", **kw)

def load_gsm8k():
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "main")
    tr = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
    ev = ds["test"].shuffle(seed=SEED).select(range(N_EVAL))
    return tr, ev

def clean_solution(a):
    body, final = a.split("####")
    body = re.sub(r"<<.*?>>", "", body).strip()
    return f"{body}\n#### {final.strip()}"

def gold_number(a):
    return a.split("####")[-1].strip().replace(",", "")

def encode(tok, question, response, add_eos=True):
    """
    Format chat prompt + assistant response.
    Mask prompt positions with -100 so loss is computed ONLY on response tokens.
    """
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True)
    resp_text = response + ("<|im_end|>\n" if add_eos else "")

    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    r_ids = tok(resp_text, add_special_tokens=False)["input_ids"]

    full_ids = (p_ids + r_ids)[:MAXLEN]
    labels = ([-100] * len(p_ids) + r_ids)[:len(full_ids)]

    # If the response itself got truncated completely by MAXLEN, preserve at least 1 response token
    if len(labels) <= len(p_ids):
        labels[-1] = full_ids[-1]

    return {"input_ids": full_ids, "labels": labels}

def extract_answer(text):
    """Returns (parsed_number_or_None, used_strict_format)."""
    m = re.findall(r"####\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        return m[-1].replace(",", ""), True
    m = re.findall(r"\\boxed\{\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        return m[-1].replace(",", ""), False
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return (nums[-1].replace(",", "") if nums else None), False

def same(a, b):
    try:
        return a is not None and abs(float(a) - float(b)) < 1e-6
    except Exception:
        return False

Writing common.py


In [3]:
%%writefile prepare_data.py
import os, json, random
from common import *

tok = get_tokenizer()
tr, ev = load_gsm8k()
rows = [{"question": r["question"], "response": clean_solution(r["answer"])} for r in tr]
os.makedirs("data", exist_ok=True)

def dump(path, rs):
    with open(path, "w") as f:
        for r in rs:
            f.write(json.dumps({"messages": [{"role": "user", "content": r["question"]},
                                              {"role": "assistant", "content": r["response"]}]}) + "\n")

# Clean data
dump("data/train.jsonl", rows)
qs = [r["question"] for r in rows]
ev_q = set(r["question"] for r in ev)
lens = [len(encode(tok, r["question"], r["response"])["input_ids"]) for r in rows]

print(f"Total train: {len(rows)} | Duplicate questions: {len(qs) - len(set(qs))}")
print(f"Train/Eval overlap: {len(set(qs) & ev_q)}")
print(f"Empty responses: {sum(1 for r in rows if not r['response'].strip())}")
print(f"Token length (mean / max): {sum(lens)/len(lens):.1f} / {max(lens)} | Truncated at MAXLEN: {sum(l >= MAXLEN for l in lens)}")

# Deliberate fault injection to test silent failure detection
rng = random.Random(SEED)
faulty = [dict(r) for r in rows]
trunc_idx = rng.sample(range(len(faulty)), k=int(0.05 * len(faulty)))
empty_idx = rng.sample(range(len(faulty)), k=int(0.02 * len(faulty)))
for i in trunc_idx:
    faulty[i]["response"] = faulty[i]["response"][: len(faulty[i]["response"]) // 2]
for i in empty_idx:
    faulty[i]["response"] = ""
faulty += [dict(r) for r in rng.sample(rows, k=int(0.03 * len(rows)))]

dump("data/train_faulty.jsonl", faulty)
manifest = {"truncated": len(trunc_idx), "empty": len(empty_idx), "duplicated": int(0.03 * len(rows))}
json.dump(manifest, open("data/train_faulty_manifest.json", "w"))
print(f"Injected faults manifest: {manifest}")

# Visual inspection of token masking
ex = encode(tok, rows[0]["question"], rows[0]["response"], add_eos=True)
supervised_tokens = [t for t in ex["labels"] if t != -100]
print("\n--- PROMPT TOKENS MASKED (-100 count) ---:", ex["labels"].count(-100))
print("--- SUPERVISED RESPONSE TOKENS COUNT ---:", len(supervised_tokens))
print("--- LAST 3 SUPERVISED TOKENS ---:", [tok.decode([t]) for t in supervised_tokens[-3:]])

Writing prepare_data.py


In [4]:
# Execute and log
!python prepare_data.py 2>&1 | tee logs/prepare_data.log

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 227263.98 examples/s]
Total train: 1500 | Duplicate questions: 0
Train/Eval overlap: 0
Empty responses: 0
Token length (mean / max): 180.2 / 428 | Truncated at MAXLEN: 0
Injected faults manifest: {'truncated': 75, 'empty': 30, 'duplicated': 45}

--- PROMPT TOKENS MASKED (-100 count) ---: 81
--- SUPERVISED RESPONSE TOKENS COUNT ---: 53
--- LAST 3 SUPERVISED TOKENS ---: ['6', '<|im_end|>', '\n']


In [5]:
%%writefile memory_budget.py
import json, os, torch
from transformers import AutoConfig, AutoModelForCausalLM
from common import *

os.makedirs("out", exist_ok=True)
GiB = 1024 ** 3
R, BS, S = 16, 4, MAXLEN

cfg = AutoConfig.from_pretrained(MODEL)
with torch.device("meta"):
    m = AutoModelForCausalLM.from_config(cfg)

P = sum(p.numel() for p in m.parameters())
H, L, I, V = cfg.hidden_size, cfg.num_hidden_layers, cfg.intermediate_size, cfg.vocab_size

# Count LoRA trainable parameters across all linear projections
targets = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
lin = [(n, mod) for n, mod in m.model.layers.named_modules() if isinstance(mod, torch.nn.Linear)]
lin_params = sum(mod.weight.numel() for _, mod in lin)
nonlin = P - lin_params
T = sum(R * (mod.in_features + mod.out_features) for n, mod in lin if n.split(".")[-1] in targets)
max_mat = max(mod.weight.numel() for _, mod in lin)

# Formulate exact theoretical budget
est = {}
est["weights_fp16"] = P * 2
est["lora_state_fp32(w+g+adam)"] = T * 16
est["ckpt_activations"] = L * BS * S * H * 2
est["one_layer_recompute"] = BS * S * (10 * H + 3 * I) * 2
est["logits(fp16+fp32+grad)"] = BS * S * V * (2 + 4 + 4)

train_total = sum(est.values())
merge_total = P * 2 + max_mat * 2 + T * 4
quant_total = lin_params * (0.5 + 4 / 64) + nonlin * 2 + max_mat * 2

print("=== ARCHITECTURE SPECS ===")
print(f"Total Params: {P/1e9:.3f}B | Linear Params: {lin_params/1e9:.3f}B | Embedding/Norms: {nonlin/1e9:.3f}B")
print(f"LoRA Target Trainable Params (r={R}): {T/1e6:.2f}M ({T/P*100:.3f}% of total)")
print(f"Hidden: {H} | Layers: {L} | Intermediate: {I} | Vocab: {V}\n")

print("=== THEORETICAL VRAM BUDGET ESTIMATES ===")
for k, v in est.items():
    print(f"  {k:30s}: {v/GiB:6.2f} GiB")
print("-" * 45)
print(f"  {'TRAIN PEAK ESTIMATE':30s}: {train_total/GiB:6.2f} GiB")
print(f"  {'MERGE PEAK ESTIMATE':30s}: {merge_total/GiB:6.2f} GiB")
print(f"  {'QUANT 4-BIT PEAK ESTIMATE':30s}: {quant_total/GiB:6.2f} GiB\n")

json.dump({
    "components_GiB": {k: v / GiB for k, v in est.items()},
    "train_GiB": train_total / GiB,
    "merge_GiB": merge_total / GiB,
    "quant_GiB": quant_total / GiB,
    "params": P,
    "lora_trainable": T
}, open("out/memory_estimate.json", "w"), indent=2)
print("Saved to out/memory_estimate.json")

Writing memory_budget.py


In [6]:
!python memory_budget.py 2>&1 | tee logs/memory_budget.log

=== ARCHITECTURE SPECS ===
Total Params: 1.544B | Linear Params: 1.310B | Embedding/Norms: 0.234B
LoRA Target Trainable Params (r=16): 18.46M (1.196% of total)
Hidden: 1536 | Layers: 28 | Intermediate: 8960 | Vocab: 151936

=== THEORETICAL VRAM BUDGET ESTIMATES ===
  weights_fp16                  :   2.88 GiB
  lora_state_fp32(w+g+adam)     :   0.28 GiB
  ckpt_activations              :   0.16 GiB
  one_layer_recompute           :   0.16 GiB
  logits(fp16+fp32+grad)        :   2.90 GiB
---------------------------------------------
  TRAIN PEAK ESTIMATE           :   6.37 GiB
  MERGE PEAK ESTIMATE           :   2.97 GiB
  QUANT 4-BIT PEAK ESTIMATE     :   1.15 GiB

Saved to out/memory_estimate.json


In [7]:
%%writefile pipeline.py
import argparse, json, os, time, gc, subprocess, datetime, torch
from datasets import Dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments, TrainerCallback, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from common import *

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True, choices=["real", "control", "noeos"])
args = ap.parse_args()

RUNS = {
    "real": dict(lr=2e-4, epochs=2, max_steps=-1, add_eos=True),
    "control": dict(lr=1e-6, epochs=1, max_steps=10, add_eos=True),
    "noeos": dict(lr=2e-4, epochs=2, max_steps=-1, add_eos=False)
}
C = RUNS[args.run]
OUT = f"out/{args.run}"
os.makedirs(OUT, exist_ok=True)
os.makedirs("logs", exist_ok=True)
seed_all(SEED)
GiB = 1024 ** 3
log(f"Starting pipeline for RUN={args.run} with config={C}")

def snap(tag):
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    with open(f"logs/nvidia_smi_{args.run}.txt", "a") as f:
        f.write(f"\n===== {datetime.datetime.now().isoformat(timespec='seconds')} {tag} =====\n{r}")

PH = {}

def phase(name):
    torch.cuda.synchronize()
    PH[name] = dict(
        max_alloc_GiB=round(torch.cuda.max_memory_allocated() / GiB, 3),
        max_reserved_GiB=round(torch.cuda.max_memory_reserved() / GiB, 3)
    )
    log(f"PHASE {name}: {PH[name]}")
    snap(name)
    torch.cuda.reset_peak_memory_stats()

def free():
    gc.collect()
    torch.cuda.empty_cache()

tok = get_tokenizer()
tr, ev = load_gsm8k()
train_ex = [encode(tok, r["question"], clean_solution(r["answer"]), add_eos=C["add_eos"]) for r in tr]
ds = Dataset.from_list(train_ex)
TOK = {"real": 0, "padded": 0}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id)
    lab = torch.full((len(batch), L), -100)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        lab[i, :n] = torch.tensor(b["labels"])
        att[i, :n] = 1
    TOK["real"] += int(att.sum())
    TOK["padded"] += ids.numel()
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

# ---------------- 1. TRAIN ----------------
torch.cuda.reset_peak_memory_stats()
model = load_model(MODEL).cuda()
phase("load_base_fp16")

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

lcfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lcfg)
n_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all_params = sum(p.numel() for p in model.parameters())
log(f"Trainable parameters: {n_train_params} of {n_all_params}")

class MetricsCallback(TrainerCallback):
    def on_train_begin(self, a, s, c, **kw):
        self.t, self.r, self.p = time.time(), TOK["real"], TOK["padded"]

    def on_log(self, a, s, c, logs=None, **kw):
        now = time.time()
        dt = max(now - self.t, 1e-9)
        dr, dp = TOK["real"] - self.r, TOK["padded"] - self.p
        self.t, self.r, self.p = now, TOK["real"], TOK["padded"]
        log(f"step={s.global_step} logs={json.dumps(logs)} real_tok/s={dr/dt:.0f} padded_tok/s={dp/dt:.0f} "
            f"alloc={torch.cuda.memory_allocated()/GiB:.2f}GiB peak_alloc={torch.cuda.max_memory_allocated()/GiB:.2f}GiB "
            f"reserved={torch.cuda.memory_reserved()/GiB:.2f}GiB")

targs = TrainingArguments(
    output_dir=f"{OUT}/trainer",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=C["lr"],
    num_train_epochs=C["epochs"],
    max_steps=C["max_steps"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=15,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    max_grad_norm=1.0,
    optim="adamw_torch"
)

trainer = Trainer(model=model, args=targs, train_dataset=ds, data_collator=collate, callbacks=[MetricsCallback()])
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_output = trainer.train()
train_secs = time.time() - t0
phase("train")

losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
model.save_pretrained(f"{OUT}/adapter")
tok.save_pretrained(f"{OUT}/adapter")
del trainer, model
free()

# ---------------- 2. MERGE ----------------
torch.cuda.reset_peak_memory_stats()
base = load_model(MODEL).cuda()
phase("merge_load_base")

pm = PeftModel.from_pretrained(base, f"{OUT}/adapter")
merged = pm.merge_and_unload()
phase("merge")

merged.save_pretrained(f"{OUT}/merged", safe_serialization=True)
tok.save_pretrained(f"{OUT}/merged")
del merged, pm, base
free()

# ---------------- 3. QUANTIZE (NF4) ----------------
torch.cuda.reset_peak_memory_stats()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
q = load_model(f"{OUT}/merged", quantization_config=bnb, device_map={"": 0})
phase("quantize_load_4bit")

q.save_pretrained(f"{OUT}/quant4")
tok.save_pretrained(f"{OUT}/quant4")
del q
free()

summary = dict(
    run=args.run,
    config=C,
    trainable_params=n_train_params,
    total_params=n_all_params,
    train_secs=train_secs,
    train_loss_mean=train_output.training_loss,
    first_logged_loss=losses[0] if losses else None,
    last_logged_loss=losses[-1] if losses else None,
    real_tokens=TOK["real"],
    padded_tokens=TOK["padded"],
    real_tok_per_s=round(TOK["real"] / train_secs, 1),
    padded_tok_per_s=round(TOK["padded"] / train_secs, 1),
    phases=PH,
    torch=torch.__version__
)

with open(f"{OUT}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

log("FINISHED RUN SUCCESSFULLY: " + json.dumps(summary, indent=2))

Writing pipeline.py


In [8]:
!python pipeline.py --run real 2>&1 | tee logs/real_pipeline.log

[2026-11-11T17:25:49] Starting pipeline for RUN=real with config={'lr': 0.0002, 'epochs': 2, 'max_steps': -1, 'add_eos': True}
Loading weights: 100%|██████████| 338/338 [00:04<00:00, 83.35it/s] 
[2026-11-11T17:26:24] PHASE load_base_fp16: {'max_alloc_GiB': 2.876, 'max_reserved_GiB': 3.061}
Traceback (most recent call last):
  File "/content/pipeline.py", line 81, in <module>
    model = get_peft_model(model, lcfg)
  File "/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py", line 198, in get_peft_model
    return MODEL_TYPE_TO_PEFT_MODEL_MAPPING[peft_config.task_type](
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        model,
        ^^^^^^
    ...<3 lines>...
        low_cpu_mem_usage=low_cpu_mem_usage,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/peft/peft_model.py", line 2082, in __init__
    super().__init__(model, peft_config, adapter_name, **kwargs)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

In [9]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [10]:
!python pipeline.py --run real 2>&1 | tee logs/real_pipeline.log

[2026-11-11T17:27:49] Starting pipeline for RUN=real with config={'lr': 0.0002, 'epochs': 2, 'max_steps': -1, 'add_eos': True}
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 100.94it/s]
[2026-11-11T17:27:58] PHASE load_base_fp16: {'max_alloc_GiB': 2.876, 'max_reserved_GiB': 3.061}
[2026-11-11T17:27:59] Trainable parameters: 18464768 of 1562179072
Traceback (most recent call last):
  File "/content/pipeline.py", line 99, in <module>
    targs = TrainingArguments(
        output_dir=f"{OUT}/trainer",
    ...<15 lines>...
        optim="adamw_torch"
    )
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'


In [11]:
!pip uninstall -y torchao
!python pipeline.py --run real 2>&1 | tee logs/real_pipeline.log

[2026-11-11T17:28:34] Starting pipeline for RUN=real with config={'lr': 0.0002, 'epochs': 2, 'max_steps': -1, 'add_eos': True}
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 144.68it/s]
[2026-11-11T17:28:41] PHASE load_base_fp16: {'max_alloc_GiB': 2.876, 'max_reserved_GiB': 3.061}
[2026-11-11T17:28:41] Trainable parameters: 18464768 of 1562179072
Traceback (most recent call last):
  File "/content/pipeline.py", line 99, in <module>
    targs = TrainingArguments(
        output_dir=f"{OUT}/trainer",
    ...<15 lines>...
        optim="adamw_torch"
    )
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'


In [12]:
%%writefile pipeline.py
import argparse, json, os, time, gc, subprocess, datetime, torch
from datasets import Dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments, TrainerCallback, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from common import *

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True, choices=["real", "control", "noeos"])
args = ap.parse_args()

RUNS = {
    "real": dict(lr=2e-4, epochs=2, max_steps=-1, add_eos=True),
    "control": dict(lr=1e-6, epochs=1, max_steps=10, add_eos=True),
    "noeos": dict(lr=2e-4, epochs=2, max_steps=-1, add_eos=False)
}
C = RUNS[args.run]
OUT = f"out/{args.run}"
os.makedirs(OUT, exist_ok=True)
os.makedirs("logs", exist_ok=True)
seed_all(SEED)
GiB = 1024 ** 3
log(f"Starting pipeline for RUN={args.run} with config={C}")

def snap(tag):
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    with open(f"logs/nvidia_smi_{args.run}.txt", "a") as f:
        f.write(f"\n===== {datetime.datetime.now().isoformat(timespec='seconds')} {tag} =====\n{r}")

PH = {}

def phase(name):
    torch.cuda.synchronize()
    PH[name] = dict(
        max_alloc_GiB=round(torch.cuda.max_memory_allocated() / GiB, 3),
        max_reserved_GiB=round(torch.cuda.max_memory_reserved() / GiB, 3)
    )
    log(f"PHASE {name}: {PH[name]}")
    snap(name)
    torch.cuda.reset_peak_memory_stats()

def free():
    gc.collect()
    torch.cuda.empty_cache()

tok = get_tokenizer()
tr, ev = load_gsm8k()
train_ex = [encode(tok, r["question"], clean_solution(r["answer"]), add_eos=C["add_eos"]) for r in tr]
ds = Dataset.from_list(train_ex)
TOK = {"real": 0, "padded": 0}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id)
    lab = torch.full((len(batch), L), -100)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        lab[i, :n] = torch.tensor(b["labels"])
        att[i, :n] = 1
    TOK["real"] += int(att.sum())
    TOK["padded"] += ids.numel()
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

# ---------------- 1. TRAIN ----------------
torch.cuda.reset_peak_memory_stats()
model = load_model(MODEL).cuda()
phase("load_base_fp16")

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

lcfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lcfg)
n_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all_params = sum(p.numel() for p in model.parameters())
log(f"Trainable parameters: {n_train_params} of {n_all_params}")

class MetricsCallback(TrainerCallback):
    def on_train_begin(self, a, s, c, **kw):
        self.t, self.r, self.p = time.time(), TOK["real"], TOK["padded"]

    def on_log(self, a, s, c, logs=None, **kw):
        now = time.time()
        dt = max(now - self.t, 1e-9)
        dr, dp = TOK["real"] - self.r, TOK["padded"] - self.p
        self.t, self.r, self.p = now, TOK["real"], TOK["padded"]
        log(f"step={s.global_step} logs={json.dumps(logs)} real_tok/s={dr/dt:.0f} padded_tok/s={dp/dt:.0f} "
            f"alloc={torch.cuda.memory_allocated()/GiB:.2f}GiB peak_alloc={torch.cuda.max_memory_allocated()/GiB:.2f}GiB "
            f"reserved={torch.cuda.memory_reserved()/GiB:.2f}GiB")

targs = TrainingArguments(
    output_dir=f"{OUT}/trainer",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=C["lr"],
    num_train_epochs=C["epochs"],
    max_steps=C["max_steps"],
    lr_scheduler_type="cosine",
    warmup_steps=10,
    fp16=True,
    logging_steps=15,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    max_grad_norm=1.0,
    optim="adamw_torch"
)

trainer = Trainer(model=model, args=targs, train_dataset=ds, data_collator=collate, callbacks=[MetricsCallback()])
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_output = trainer.train()
train_secs = time.time() - t0
phase("train")

losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
model.save_pretrained(f"{OUT}/adapter")
tok.save_pretrained(f"{OUT}/adapter")
del trainer, model
free()

# ---------------- 2. MERGE ----------------
torch.cuda.reset_peak_memory_stats()
base = load_model(MODEL).cuda()
phase("merge_load_base")

pm = PeftModel.from_pretrained(base, f"{OUT}/adapter")
merged = pm.merge_and_unload()
phase("merge")

merged.save_pretrained(f"{OUT}/merged", safe_serialization=True)
tok.save_pretrained(f"{OUT}/merged")
del merged, pm, base
free()

# ---------------- 3. QUANTIZE (NF4) ----------------
torch.cuda.reset_peak_memory_stats()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
q = load_model(f"{OUT}/merged", quantization_config=bnb, device_map={"": 0})
phase("quantize_load_4bit")

q.save_pretrained(f"{OUT}/quant4")
tok.save_pretrained(f"{OUT}/quant4")
del q
free()

summary = dict(
    run=args.run,
    config=C,
    trainable_params=n_train_params,
    total_params=n_all_params,
    train_secs=train_secs,
    train_loss_mean=train_output.training_loss,
    first_logged_loss=losses[0] if losses else None,
    last_logged_loss=losses[-1] if losses else None,
    real_tokens=TOK["real"],
    padded_tokens=TOK["padded"],
    real_tok_per_s=round(TOK["real"] / train_secs, 1),
    padded_tok_per_s=round(TOK["padded"] / train_secs, 1),
    phases=PH,
    torch=torch.__version__
)

with open(f"{OUT}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

log("FINISHED RUN SUCCESSFULLY: " + json.dumps(summary, indent=2))

Overwriting pipeline.py


In [13]:
!python pipeline.py --run real 2>&1 | tee logs/real_pipeline.log

[2026-11-11T17:30:20] Starting pipeline for RUN=real with config={'lr': 0.0002, 'epochs': 2, 'max_steps': -1, 'add_eos': True}
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 132.00it/s]
[2026-11-11T17:30:27] PHASE load_base_fp16: {'max_alloc_GiB': 2.876, 'max_reserved_GiB': 3.061}
[2026-11-11T17:30:28] Trainable parameters: 18464768 of 1562179072
 16%|█▌        | 30/188 [01:32<08:22,  3.18s/it]{'loss': '0.563', 'grad_norm': '0.4346', 'learning_rate': '0.0001998', 'epoch': '0.16'}
[2026-11-11T17:32:01] step=30 logs={"loss": 0.4235862096150716, "grad_norm": 0.40231797099113464, "learning_rate": 0.00019442989534495557, "epoch": 0.32} real_tok/s=912 padded_tok/s=1208 alloc=3.08GiB peak_alloc=5.74GiB reserved=6.09GiB
 24%|██▍       | 45/188 [02:21<07:49,  3.28s/it]{'loss': '0.4236', 'grad_norm': '0.4023', 'learning_rate': '0.0001944', 'epoch': '0.32'}
[2026-11-11T17:32:49] step=45 logs={"loss": 0.4508102734883626, "grad_norm": 0.40281593799591064, "learning_rate": 0.000182529072569

In [14]:
# 1. Run the control pipeline (takes ~35 seconds)
!python pipeline.py --run control 2>&1 | tee logs/control_pipeline.log

# 2. Run the noeos pipeline (takes ~10 minutes)
!python pipeline.py --run noeos 2>&1 | tee logs/noeos_pipeline.log

[2026-11-11T17:50:57] Starting pipeline for RUN=control with config={'lr': 1e-06, 'epochs': 1, 'max_steps': 10, 'add_eos': True}
Loading weights: 100%|██████████| 338/338 [00:10<00:00, 31.45it/s]
[2026-11-11T17:51:13] PHASE load_base_fp16: {'max_alloc_GiB': 2.876, 'max_reserved_GiB': 3.061}
[2026-11-11T17:51:14] Trainable parameters: 18464768 of 1562179072
100%|██████████| 10/10 [00:31<00:00,  3.15s/it]
{'train_runtime': '31.47', 'train_samples_per_second': '5.084', 'train_steps_per_second': '0.318', 'train_loss': '0.8595', 'epoch': '0.1067'}
[2026-11-11T17:51:46] PHASE train: {'max_alloc_GiB': 5.645, 'max_reserved_GiB': 12.457}
Loading weights: 100%|██████████| 338/338 [00:05<00:00, 61.84it/s]
[2026-11-11T17:51:53] PHASE merge_load_base: {'max_alloc_GiB': 2.876, 'max_reserved_GiB': 3.061}
[2026-11-11T17:51:55] PHASE merge: {'max_alloc_GiB': 3.047, 'max_reserved_GiB': 3.314}
Loading weights: 100%|██████████| 338/338 [00:09<00:00, 34.35it/s]
[2026-11-11T17:55:24] PHASE quantize_load_4bi

In [15]:
%%writefile verify.py
import argparse, json, os, gc, numpy as np, torch
from safetensors.torch import load_file
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig
from common import *

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True)
ap.add_argument("--n_gen", type=int, default=N_EVAL)
args = ap.parse_args()

OUT = f"out/{args.run}"
tok = get_tokenizer()
tr, ev = load_gsm8k()
ev_pairs = [(r["question"], clean_solution(r["answer"])) for r in ev]
ev_gold = [gold_number(r["answer"]) for r in ev]
gen_idx = list(range(min(args.n_gen, len(ev_pairs))))

# General-ability benchmark: Alpaca Cleaned to measure catastrophic forgetting
gp = []
for r in load_dataset("yahma/alpaca-cleaned")["train"].shuffle(seed=SEED):
    if 0 < len(r["output"]) < 1000:
        gp.append((r["instruction"] + ("\n\n" + r["input"] if r["input"] else ""), r["output"]))
    if len(gp) == 100:
        break

log("Loading base, merged fp16, and direct 4-bit")
base = load_model(MODEL).cuda().eval()
merged = load_model(f"{OUT}/merged").cuda().eval()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
q4 = load_model(f"{OUT}/merged", quantization_config=bnb, device_map={"": 0}).eval()

# 1. Adapter & Merge Audit
ad = load_file(f"{OUT}/adapter/adapter_model.safetensors")
acfg = json.load(open(f"{OUT}/adapter/adapter_config.json"))
scale = acfg["lora_alpha"] / acfg["r"]
bsd, msd = base.state_dict(), merged.state_dict()
rows = []
for k in ad:
    if "lora_A" not in k:
        continue
    A = ad[k].float().cuda()
    B = ad[k.replace("lora_A", "lora_B")].float().cuda()
    wk = k.replace("base_model.model.", "", 1).replace(".lora_A.weight", ".weight")
    if wk not in bsd:
        continue
    dW = scale * (B @ A)
    W, Wm = bsd[wk].float(), msd[wk].float()
    diff = Wm - W
    rows.append(dict(
        layer=wk,
        B_absmax=float(B.abs().max()),
        rel_dW=float(dW.norm() / W.norm()),
        merge_err=float((diff - dW).norm() / dW.norm().clamp(min=1e-30)),
        identical=bool(torch.equal(Wm.half(), W.half()))
    ))

audit = dict(
    n_lora_layers=len(rows),
    B_nonzero_layers=sum(r["B_absmax"] > 0 for r in rows),
    mean_rel_dW=float(np.mean([r["rel_dW"] for r in rows])),
    max_rel_dW=float(np.max([r["rel_dW"] for r in rows])),
    merged_identical_to_base_layers=sum(r["identical"] for r in rows),
    mean_merge_err=float(np.mean([r["merge_err"] for r in rows]))
)
log("AUDIT SUMMARY: " + json.dumps(audit))

# Helper evaluation functions
def batchify(pairs):
    exs = [encode(tok, q, r) for q, r in pairs]
    L = max(len(e["input_ids"]) for e in exs)
    ids = torch.full((len(exs), L), tok.pad_token_id)
    lab = torch.full((len(exs), L), -100)
    att = torch.zeros((len(exs), L), dtype=torch.long)
    for j, e in enumerate(exs):
        n = len(e["input_ids"])
        ids[j, :n] = torch.tensor(e["input_ids"])
        lab[j, :n] = torch.tensor(e["labels"])
        att[j, :n] = 1
    return ids, lab, att

@torch.no_grad()
def per_example_nll(m, pairs, bs=4):
    res = []
    for i in range(0, len(pairs), bs):
        ids, lab, att = batchify(pairs[i:i + bs])
        logits = m(input_ids=ids.cuda(), attention_mask=att.cuda()).logits[:, :-1].float()
        tgt = lab[:, 1:].cuda()
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction="none").view(tgt.shape)
        mask = tgt != -100
        res += ((loss * mask).sum(1) / mask.sum(1).clamp(min=1)).tolist()
    return res

@torch.no_grad()
def mean_kl(ref, other, pairs, bs=2):
    tot, cnt = 0.0, 0
    for i in range(0, len(pairs), bs):
        ids, lab, att = batchify(pairs[i:i + bs])
        lr = ref(input_ids=ids.cuda(), attention_mask=att.cuda()).logits[:, :-1].float().log_softmax(-1)
        lo = other(input_ids=ids.cuda(), attention_mask=att.cuda()).logits[:, :-1].float().log_softmax(-1)
        mask = (lab[:, 1:] != -100).cuda()
        kl = (lr.exp() * (lr - lo)).sum(-1)
        tot += float((kl * mask).sum())
        cnt += int(mask.sum())
    return tot / max(cnt, 1)

@torch.no_grad()
def generate_batch(m, questions, bs=16):
    tok.padding_side = "left"
    stop_ids = {tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")}
    texts, lens, ended = [], [], []
    t0 = time.time()
    for i in range(0, len(questions), bs):
        prompts = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in questions[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).cuda()
        out = m.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        for g in out[:, enc["input_ids"].shape[1]:].tolist():
            n = next((j for j, t in enumerate(g) if t in stop_ids), None)
            ended.append(n is not None)
            n = len(g) if n is None else n
            lens.append(n)
            texts.append(tok.decode(g[:n], skip_special_tokens=True))
    torch.cuda.synchronize()
    return dict(texts=texts, lens=lens, ended=ended, secs=time.time() - t0)

def evaluate(m):
    r = {"nll": per_example_nll(m, ev_pairs), "general_nll": per_example_nll(m, gp)}
    g = generate_batch(m, [ev_pairs[i][0] for i in gen_idx])
    correct, strict = [], []
    for t, i in zip(g["texts"], gen_idx):
        a, s = extract_answer(t)
        correct.append(int(same(a, ev_gold[i])))
        strict.append(int(s))
    r.update(correct=correct, strict=strict, ended=[int(e) for e in g["ended"]], lens=g["lens"],
             tok_per_s=sum(g["lens"]) / g["secs"],
             samples=[dict(q=ev_pairs[i][0], gold=ev_gold[i], out=g["texts"][j]) for j, i in enumerate(gen_idx[:3])])
    return r

R = {}
cache = "out/base_eval.json"
if os.path.exists(cache) and json.load(open(cache)).get("n_gen") == len(gen_idx):
    R["base"] = json.load(open(cache))
    log("Base evaluation loaded from cache")
else:
    log("Evaluating Base Model")
    R["base"] = evaluate(base)
    R["base"]["n_gen"] = len(gen_idx)
    json.dump(R["base"], open(cache, "w"))

log("Evaluating Merged Model")
R["merged"] = evaluate(merged)
log("Evaluating 4-Bit Model")
R["q4"] = evaluate(q4)

kl_pairs = ev_pairs[:100]
KL = dict(
    base_to_merged=mean_kl(base, merged, kl_pairs),
    merged_to_q4=mean_kl(merged, q4, kl_pairs)
)

def boot(d, n=5000, seed=0):
    d = np.asarray(d, dtype=float)
    m = np.random.default_rng(seed).choice(d, (n, len(d)), replace=True).mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def summ(r):
    return dict(nll=float(np.mean(r["nll"])), general_nll=float(np.mean(r["general_nll"])),
                acc=float(np.mean(r["correct"])), strict_format=float(np.mean(r["strict"])),
                stop_rate=float(np.mean(r["ended"])), mean_len=float(np.mean(r["lens"])),
                tok_per_s=round(r["tok_per_s"], 1))

S_ = {k: summ(v) for k, v in R.items()}

cmp = {}
cmp["nll_improve_base_minus_merged_CI95"] = boot(np.array(R["base"]["nll"]) - np.array(R["merged"]["nll"]))
cmp["general_nll_increase_merged_minus_base_CI95"] = boot(np.array(R["merged"]["general_nll"]) - np.array(R["base"]["general_nll"]))
cmp["acc_merged_minus_base_CI95"] = boot(np.array(R["merged"]["correct"]) - np.array(R["base"]["correct"]))
cmp["q4_nll_minus_merged_CI95"] = boot(np.array(R["q4"]["nll"]) - np.array(R["merged"]["nll"]))

weights_moved = audit["B_nonzero_layers"] > 0 and audit["merged_identical_to_base_layers"] < audit["n_lora_layers"]
nll_better = cmp["nll_improve_base_minus_merged_CI95"][0] > 0
shift_over_noise = KL["base_to_merged"] > KL["merged_to_q4"]

gates = dict(
    weights_moved=weights_moved,
    heldout_nll_significantly_better=nll_better,
    behaviour_shift_exceeds_quant_noise=shift_over_noise,
    merged_stops_generating=S_["merged"]["stop_rate"] >= 0.95,
    q4_stops_generating=S_["q4"]["stop_rate"] >= 0.95,
    q4_not_significantly_worse_nll=cmp["q4_nll_minus_merged_CI95"][0] <= 0
)
gates["VERDICT_model_actually_trained"] = weights_moved and nll_better and shift_over_noise

res = dict(run=args.run, audit=audit, summary=S_, KL=KL, compare=cmp, gates=gates,
           samples={k: v["samples"] for k, v in R.items()})
json.dump(res, open(f"{OUT}/verify.json", "w"), indent=2)

print("\n" + "=" * 30 + f" VERIFY RESULT: {args.run} " + "=" * 30)
print(json.dumps(dict(gates=gates, KL=KL, compare=cmp), indent=2))
print(f"\n{'Artifact':<10} {'NLL':>7} {'Gen_NLL':>9} {'Acc':>7} {'Strict':>8} {'StopRate':>10} {'Tok/s':>7}")
for k, v in S_.items():
    print(f"{k:<10} {v['nll']:7.3f} {v['general_nll']:9.3f} {v['acc']:7.3f} {v['strict_format']:8.3f} {v['stop_rate']:10.3f} {v['tok_per_s']:7.1f}")

Writing verify.py


In [16]:
!python verify.py --run real 2>&1 | tee logs/verify_real.log
!python verify.py --run control --n_gen 100 2>&1 | tee logs/verify_control.log
!python verify.py --run noeos --n_gen 100 2>&1 | tee logs/verify_noeos.log

Generating train split: 100%|██████████| 51760/51760 [00:00<00:00, 65350.97 examples/s]
[2026-11-11T18:14:03] Loading base, merged fp16, and direct 4-bit
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 86.91it/s] 
[2026-11-11T18:14:36] AUDIT SUMMARY: {"n_lora_layers": 196, "B_nonzero_layers": 196, "mean_rel_dW": 0.007538293301821591, "max_rel_dW": 0.012394731864333153, "merged_identical_to_base_layers": 0, "mean_merge_err": 0.02950243032252302}
[2026-11-11T18:14:36] Evaluating Base Model
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py", line 293, in __getattr__
    return self.data[item]
           ~~~~~~~~~^^^^^^
KeyError: 'cuda'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/verify.py", line 152, in <module>
    R["base"] = evaluate(base)
                ~~~~~~~~^^^^^^
  File "/content/verify.py", line 134, in evaluate
    g = g

In [19]:
%%writefile verify.py
import argparse, json, os, gc, numpy as np, torch
from safetensors.torch import load_file
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig
from common import *

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True)
ap.add_argument("--n_gen", type=int, default=N_EVAL)
args = ap.parse_args()

OUT = f"out/{args.run}"
tok = get_tokenizer()
tr, ev = load_gsm8k()
ev_pairs = [(r["question"], clean_solution(r["answer"])) for r in ev]
ev_gold = [gold_number(r["answer"]) for r in ev]
gen_idx = list(range(min(args.n_gen, len(ev_pairs))))

# General-ability benchmark: Alpaca Cleaned to measure catastrophic forgetting
gp = []
for r in load_dataset("yahma/alpaca-cleaned")["train"].shuffle(seed=SEED):
    if 0 < len(r["output"]) < 1000:
        gp.append((r["instruction"] + ("\n\n" + r["input"] if r["input"] else ""), r["output"]))
    if len(gp) == 100:
        break

log("Loading base, merged fp16, and direct 4-bit")
base = load_model(MODEL).cuda().eval()
merged = load_model(f"{OUT}/merged").cuda().eval()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
q4 = load_model(f"{OUT}/merged", quantization_config=bnb, device_map={"": 0}).eval()

# 1. Adapter & Merge Audit
ad = load_file(f"{OUT}/adapter/adapter_model.safetensors")
acfg = json.load(open(f"{OUT}/adapter/adapter_config.json"))
scale = acfg["lora_alpha"] / acfg["r"]
bsd, msd = base.state_dict(), merged.state_dict()
rows = []
for k in ad:
    if "lora_A" not in k:
        continue
    A = ad[k].float().cuda()
    B = ad[k.replace("lora_A", "lora_B")].float().cuda()
    wk = k.replace("base_model.model.", "", 1).replace(".lora_A.weight", ".weight")
    if wk not in bsd:
        continue
    dW = scale * (B @ A)
    W, Wm = bsd[wk].float(), msd[wk].float()
    diff = Wm - W
    rows.append(dict(
        layer=wk,
        B_absmax=float(B.abs().max()),
        rel_dW=float(dW.norm() / W.norm()),
        merge_err=float((diff - dW).norm() / dW.norm().clamp(min=1e-30)),
        identical=bool(torch.equal(Wm.half(), W.half()))
    ))

audit = dict(
    n_lora_layers=len(rows),
    B_nonzero_layers=sum(r["B_absmax"] > 0 for r in rows),
    mean_rel_dW=float(np.mean([r["rel_dW"] for r in rows])),
    max_rel_dW=float(np.max([r["rel_dW"] for r in rows])),
    merged_identical_to_base_layers=sum(r["identical"] for r in rows),
    mean_merge_err=float(np.mean([r["merge_err"] for r in rows]))
)
log("AUDIT SUMMARY: " + json.dumps(audit))

def batchify(pairs):
    exs = [encode(tok, q, r) for q, r in pairs]
    L = max(len(e["input_ids"]) for e in exs)
    ids = torch.full((len(exs), L), tok.pad_token_id)
    lab = torch.full((len(exs), L), -100)
    att = torch.zeros((len(exs), L), dtype=torch.long)
    for j, e in enumerate(exs):
        n = len(e["input_ids"])
        ids[j, :n] = torch.tensor(e["input_ids"])
        lab[j, :n] = torch.tensor(e["labels"])
        att[j, :n] = 1
    return ids, lab, att

@torch.no_grad()
def per_example_nll(m, pairs, bs=4):
    res = []
    for i in range(0, len(pairs), bs):
        ids, lab, att = batchify(pairs[i:i + bs])
        logits = m(input_ids=ids.cuda(), attention_mask=att.cuda()).logits[:, :-1].float()
        tgt = lab[:, 1:].cuda()
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction="none").view(tgt.shape)
        mask = tgt != -100
        res += ((loss * mask).sum(1) / mask.sum(1).clamp(min=1)).tolist()
    return res

@torch.no_grad()
def mean_kl(ref, other, pairs, bs=2):
    tot, cnt = 0.0, 0
    for i in range(0, len(pairs), bs):
        ids, lab, att = batchify(pairs[i:i + bs])
        lr = ref(input_ids=ids.cuda(), attention_mask=att.cuda()).logits[:, :-1].float().log_softmax(-1)
        lo = other(input_ids=ids.cuda(), attention_mask=att.cuda()).logits[:, :-1].float().log_softmax(-1)
        mask = (lab[:, 1:] != -100).cuda()
        kl = (lr.exp() * (lr - lo)).sum(-1)
        tot += float((kl * mask).sum())
        cnt += int(mask.sum())
    return tot / max(cnt, 1)

@torch.no_grad()
def generate_batch(m, questions, bs=16):
    tok.padding_side = "left"
    stop_ids = {tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")}
    texts, lens, ended = [], [], []
    t0 = time.time()
    for i in range(0, len(questions), bs):
        prompts = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in questions[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = m.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        for g in out[:, enc["input_ids"].shape[1]:].tolist():
            n = next((j for j, t in enumerate(g) if t in stop_ids), None)
            ended.append(n is not None)
            n = len(g) if n is None else n
            lens.append(n)
            texts.append(tok.decode(g[:n], skip_special_tokens=True))
    torch.cuda.synchronize()
    return dict(texts=texts, lens=lens, ended=ended, secs=time.time() - t0)

def evaluate(m):
    r = {"nll": per_example_nll(m, ev_pairs), "general_nll": per_example_nll(m, gp)}
    g = generate_batch(m, [ev_pairs[i][0] for i in gen_idx])
    correct, strict = [], []
    for t, i in zip(g["texts"], gen_idx):
        a, s = extract_answer(t)
        correct.append(int(same(a, ev_gold[i])))
        strict.append(int(s))
    r.update(correct=correct, strict=strict, ended=[int(e) for e in g["ended"]], lens=g["lens"],
             tok_per_s=sum(g["lens"]) / max(g["secs"], 1e-6),
             samples=[dict(q=ev_pairs[i][0], gold=ev_gold[i], out=g["texts"][j]) for j, i in enumerate(gen_idx[:3])])
    return r

R = {}
cache = "out/base_eval.json"
if os.path.exists(cache) and json.load(open(cache)).get("n_gen") == len(gen_idx):
    R["base"] = json.load(open(cache))
    log("Base evaluation loaded from cache")
else:
    log("Evaluating Base Model")
    R["base"] = evaluate(base)
    R["base"]["n_gen"] = len(gen_idx)
    json.dump(R["base"], open(cache, "w"))

log("Evaluating Merged Model")
R["merged"] = evaluate(merged)
log("Evaluating 4-Bit Model")
R["q4"] = evaluate(q4)

kl_pairs = ev_pairs[:100]
KL = dict(
    base_to_merged=mean_kl(base, merged, kl_pairs),
    merged_to_q4=mean_kl(merged, q4, kl_pairs)
)

def boot(d, n=5000, seed=0):
    d = np.asarray(d, dtype=float)
    m = np.random.default_rng(seed).choice(d, (n, len(d)), replace=True).mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def summ(r):
    return dict(nll=float(np.mean(r["nll"])), general_nll=float(np.mean(r["general_nll"])),
                acc=float(np.mean(r["correct"])), strict_format=float(np.mean(r["strict"])),
                stop_rate=float(np.mean(r["ended"])), mean_len=float(np.mean(r["lens"])),
                tok_per_s=round(r["tok_per_s"], 1))

S_ = {k: summ(v) for k, v in R.items()}

cmp = {}
cmp["nll_improve_base_minus_merged_CI95"] = boot(np.array(R["base"]["nll"]) - np.array(R["merged"]["nll"]))
cmp["general_nll_increase_merged_minus_base_CI95"] = boot(np.array(R["merged"]["general_nll"]) - np.array(R["base"]["general_nll"]))
cmp["acc_merged_minus_base_CI95"] = boot(np.array(R["merged"]["correct"]) - np.array(R["base"]["correct"]))
cmp["q4_nll_minus_merged_CI95"] = boot(np.array(R["q4"]["nll"]) - np.array(R["merged"]["nll"]))

weights_moved = audit["B_nonzero_layers"] > 0 and audit["merged_identical_to_base_layers"] < audit["n_lora_layers"]
nll_better = cmp["nll_improve_base_minus_merged_CI95"][0] > 0
shift_over_noise = KL["base_to_merged"] > KL["merged_to_q4"]

gates = dict(
    weights_moved=weights_moved,
    heldout_nll_significantly_better=nll_better,
    behaviour_shift_exceeds_quant_noise=shift_over_noise,
    merged_stops_generating=S_["merged"]["stop_rate"] >= 0.95,
    q4_stops_generating=S_["q4"]["stop_rate"] >= 0.95,
    q4_not_significantly_worse_nll=cmp["q4_nll_minus_merged_CI95"][0] <= 0
)
gates["VERDICT_model_actually_trained"] = weights_moved and nll_better and shift_over_noise

res = dict(run=args.run, audit=audit, summary=S_, KL=KL, compare=cmp, gates=gates,
           samples={k: v["samples"] for k, v in R.items()})
json.dump(res, open(f"{OUT}/verify.json", "w"), indent=2)

print("\n" + "=" * 30 + f" VERIFY RESULT: {args.run} " + "=" * 30)
print(json.dumps(dict(gates=gates, KL=KL, compare=cmp), indent=2))
print(f"\n{'Artifact':<10} {'NLL':>7} {'Gen_NLL':>9} {'Acc':>7} {'Strict':>8} {'StopRate':>10} {'Tok/s':>7}")
for k, v in S_.items():
    print(f"{k:<10} {v['nll']:7.3f} {v['general_nll']:9.3f} {v['acc']:7.3f} {v['strict_format']:8.3f} {v['stop_rate']:10.3f} {v['tok_per_s']:7.1f}")

Overwriting verify.py


In [20]:
!python verify.py --run real 2>&1 | tee logs/verify_real.log
!python verify.py --run control --n_gen 100 2>&1 | tee logs/verify_control.log
!python verify.py --run noeos --n_gen 100 2>&1 | tee logs/verify_noeos.log

[2026-11-11T18:19:05] Loading base, merged fp16, and direct 4-bit
Loading weights: 100%|██████████| 338/338 [00:12<00:00, 27.80it/s]
[2026-11-11T18:19:49] AUDIT SUMMARY: {"n_lora_layers": 196, "B_nonzero_layers": 196, "mean_rel_dW": 0.007538293301821591, "max_rel_dW": 0.012394731864333153, "merged_identical_to_base_layers": 0, "mean_merge_err": 0.02950243032252302}
[2026-11-11T18:19:49] Evaluating Base Model
[2026-11-11T18:23:36] Evaluating Merged Model
[2026-11-11T18:25:36] Evaluating 4-Bit Model

============================== VERIFY RESULT: real ==============================
{
  "gates": {
    "weights_moved": true,
    "heldout_nll_significantly_better": true,
    "behaviour_shift_exceeds_quant_noise": true,
    "merged_stops_generating": true,
    "q4_stops_generating": true,
    "q4_not_significantly_worse_nll": false,
    "VERDICT_model_actually_trained": true
  },
  "KL": {
    "base_to_merged": 0.4440662558968711,
    "merged_to_q4": 0.07087084390584705
  },
  "compare": {
  

In [21]:
import os
from common import MODEL, load_model, get_tokenizer

SOUP = "/content/soupenv/bin/soup"

# Export base model to local disk for soup CLI
if not os.path.exists("base/model.safetensors"):
    print("Exporting local base model for soup...")
    m = load_model(MODEL)
    m.save_pretrained("base")
    tok = get_tokenizer()
    tok.save_pretrained("base")
    del m
    print("Base model exported to ./base")

# Check Soup CLI help and available subcommands
!{SOUP} --help 2>&1 | tee logs/soup_help.txt
!{SOUP} doctor 2>&1 | tee logs/soup_doctor.txt
!{SOUP} data --help 2>&1 | tee logs/soup_data_help.txt
!{SOUP} ship --help 2>&1 | tee logs/soup_ship_help.txt

Exporting local base model for soup...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Base model exported to ./base
                                                                                
 Usage: soup [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
 Fine-tune and post-train LLMs in one command. No SSH, no config hell.          
                                                                                
 GitHub: https://github.com/MakazhanAlpamys/Soup                                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --version                            Show Soup CLI version.                  │
│ --verbose             -V             Show full traceback on errors           │
│ --log-level                   <str>  Logging tier: quiet | normal | verbose  │
│                                      | debug                                 

In [22]:
import json
from common import load_gsm8k, clean_solution, gold_number

SOUP = "/content/soupenv/bin/soup"

# 1. Prepare evaluation set for `soup ship`
_, ev = load_gsm8k()
with open("data/soup_task_eval.jsonl", "w") as f:
    for r in ev:
        f.write(json.dumps({
            "prompt": [{"role": "user", "content": r["question"]}],
            "reference": clean_solution(r["answer"]),
            "gold": gold_number(r["answer"])
        }) + "\n")
print("Saved data/soup_task_eval.jsonl with", len(ev), "eval tasks.")

# 2. Run Soup data checks on CLEAN dataset
print("\n" + "="*25 + " SOUP DATA CHECKS: CLEAN DATA " + "="*25)
!{SOUP} data validate data/train.jsonl 2>&1 | tee logs/soup_validate_clean.txt; echo "exit=$?"
!{SOUP} data inspect data/train.jsonl 2>&1 | tee logs/soup_inspect_clean.txt

# 3. Run Soup data checks on FAULTY dataset (Silent Fault Injection Test)
print("\n" + "="*25 + " SOUP DATA CHECKS: FAULTY DATA " + "="*25)
!{SOUP} data validate data/train_faulty.jsonl 2>&1 | tee logs/soup_validate_faulty.txt; echo "exit=$?"
!{SOUP} data inspect data/train_faulty.jsonl 2>&1 | tee logs/soup_inspect_faulty.txt

# 4. Run `soup ship` against all three adapters
print("\n" + "="*25 + " SOUP SHIP: REAL ADAPTER " + "="*25)
!{SOUP} ship --base ./base --adapter out/real/adapter --task-eval data/soup_task_eval.jsonl --device cuda 2>&1 | tee logs/soup_ship_real.txt; echo "exit=$?"

print("\n" + "="*25 + " SOUP SHIP: CONTROL ADAPTER " + "="*25)
!{SOUP} ship --base ./base --adapter out/control/adapter --task-eval data/soup_task_eval.jsonl --device cuda 2>&1 | tee logs/soup_ship_control.txt; echo "exit=$?"

print("\n" + "="*25 + " SOUP SHIP: NOEOS ADAPTER " + "="*25)
!{SOUP} ship --base ./base --adapter out/noeos/adapter --task-eval data/soup_task_eval.jsonl --device cuda 2>&1 | tee logs/soup_ship_noeos.txt; echo "exit=$?"

Saved data/soup_task_eval.jsonl with 200 eval tasks.

========================= SOUP DATA CHECKS: CLEAN DATA =========================
Auto-detected format: chatml
Dataset is valid!

1500/1500 rows valid for chatml format
exit=0
Inspecting data/train.jsonl...

          Dataset Stats          
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric             ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ Total samples      │ 1500     │
│ Columns            │ messages │
│ Avg length (chars) │ 553      │
│ Min length         │ 177      │
│ Max length         │ 1404     │
│ Empty fields       │ 0        │
│ Duplicates         │ 0        │
└────────────────────┴──────────┘

Sample rows (5):
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ messages                                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ [{'role': 'user', 'content': 'Mimi picked up 2 dozen seashel │
├────────────────────────────────────────────────────

In [24]:
import json

print("=" * 35 + " 1. MEMORY BUDGET: ESTIMATE VS MEASURED " + "=" * 35)
est = json.load(open("out/memory_estimate.json"))
real_sum = json.load(open("out/real/summary.json"))
ctrl_sum = json.load(open("out/control/summary.json"))
noeos_sum = json.load(open("out/noeos/summary.json"))

print(f"{'Phase':<22} | {'Estimated (GiB)':<15} | {'Peak Alloc (GiB)':<16} | {'Peak Reserved (GiB)':<18}")
print("-" * 80)
print(f"{'Load Base (fp16)':<22} | {est['components_GiB']['weights_fp16']:<15.2f} | {real_sum['phases']['load_base_fp16']['max_alloc_GiB']:<16.3f} | {real_sum['phases']['load_base_fp16']['max_reserved_GiB']:<18.3f}")
print(f"{'LoRA Training Peak':<22} | {est['train_GiB']:<15.2f} | {real_sum['phases']['train']['max_alloc_GiB']:<16.3f} | {real_sum['phases']['train']['max_reserved_GiB']:<18.3f}")
print(f"{'Merge Peak':<22} | {est['merge_GiB']:<15.2f} | {real_sum['phases']['merge']['max_alloc_GiB']:<16.3f} | {real_sum['phases']['merge']['max_reserved_GiB']:<18.3f}")
print(f"{'4-Bit NF4 Quantize':<22} | {est['quant_GiB']:<15.2f} | {real_sum['phases']['quantize_load_4bit']['max_alloc_GiB']:<16.3f} | {real_sum['phases']['quantize_load_4bit']['max_reserved_GiB']:<18.3f}")

print("\n" + "=" * 35 + " 2. THROUGHPUT & LOSS PROFILE " + "=" * 35)
for s in [real_sum, ctrl_sum, noeos_sum]:
    print(f"Run: {s['run']:<8} | Train Secs: {s['train_secs']:<6.1f} | Real tok/s: {s['real_tok_per_s']:<6.1f} | Padded tok/s: {s['padded_tok_per_s']:<6.1f} | First Loss: {str(s['first_logged_loss'])[:6]} | Last Loss: {str(s['last_logged_loss'])[:6]}")

print("\n" + "=" * 35 + " 3. VERIFICATION METRICS ACROSS ARTIFACTS " + "=" * 35)
for run in ["real", "control", "noeos"]:
    v = json.load(open(f"out/{run}/verify.json"))
    print(f"\n--- RUN: {run.upper()} ---")
    print(f"{'Artifact':<10} | {'NLL':>7} | {'Gen NLL':>8} | {'Accuracy':>8} | {'Strict %':>8} | {'Stop Rate':>9} | {'Eval Tok/s':>10}")
    print("-" * 75)
    for art, m in v["summary"].items():
        print(f"{art:<10} | {m['nll']:>7.3f} | {m['general_nll']:>8.3f} | {m['acc']:>8.3f} | {m['strict_format']:>8.3f} | {m['stop_rate']:>9.3f} | {m['tok_per_s']:>10.1f}")

=================================== 1. MEMORY BUDGET: ESTIMATE VS MEASURED ===================================
Phase                  | Estimated (GiB) | Peak Alloc (GiB) | Peak Reserved (GiB)
--------------------------------------------------------------------------------
Load Base (fp16)       | 2.88            | 2.876            | 3.061             
LoRA Training Peak     | 6.37            | 6.206            | 14.104            
Merge Peak             | 2.97            | 3.047            | 3.314             
4-Bit NF4 Quantize     | 1.15            | 1.153            | 1.176             

=================================== 2. THROUGHPUT & LOSS PROFILE ===================================
Run: real     | Train Secs: 594.1  | Real tok/s: 909.8  | Padded tok/s: 1204.6 | First Loss: 0.5630 | Last Loss: 0.3200
Run: control  | Train Secs: 31.9   | Real tok/s: 920.0  | Padded tok/s: 1247.1 | First Loss: None | Last Loss: None
Run: noeos    | Train Secs: 594.9  | Real tok/s: 898.5  | Padded